# Week 2.2: How Much Can Your Twin Hold?
### Plain-English edition — no coding required

> **Why can't you just paste a 500-page book into the prompt?**

Because a model has no memory at all. Every single time you press send, you re-send
**everything** it is allowed to know — the instructions, the whole conversation so far,
and your new question. That bundle has a hard size limit called the **context window**.

When the bundle no longer fits, something has to be thrown away. This notebook is about
choosing **what** gets thrown away, instead of letting it happen by accident.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Tokens** — the unit everything is measured and charged in
- **Counting** — estimating the size of any piece of text before you send it
- **The window** — what fits, what overflows, and what it costs
- **The growing conversation** — why a long chat hits the wall on its own
- **Coping strategies** — trimming, summarising and sliding windows
- **Your policy** — deciding what your twin must never forget

---
## Part 1: Measuring text in tokens

Models do not count words or letters. They count **tokens** — chunks of roughly four
characters. "Hello" is one token. "Extraordinarily" might be three. The handy rule:

> **characters ÷ 4 ≈ tokens**, or **words × 1.3 ≈ tokens**

In [ ]:
#@title ▶ Step 1 — Measure a piece of my text { display-mode: "form" }
#@markdown Type or paste anything. Long is fine — that is the point.
my_text = "I run every morning before work. My best 5km time is 25 minutes and I am training for a 10km race in March." #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

text = my_text.strip()
if len(text) < 5:
    problem("Please type something longer, then press play again.")
else:
    chars = len(text); words = len(text.split())
    by_chars = count_tokens(text); by_words = max(1, round(words * 1.3))
    show_text("Your text:", text)
    print("")
    print("   characters : " + str(chars))
    print("   words      : " + str(words))
    print("")
    print("Two ways to estimate the tokens:")
    print("   characters / 4  ->  " + str(by_chars) + " tokens")
    print("   words x 1.3     ->  " + str(by_words) + " tokens")
    print("")
    gap = abs(by_chars - by_words)
    if gap <= max(2, by_chars * 0.25):
        success("Both estimates agree within " + str(gap) + " tokens, so about "
                + str(by_chars) + " is a safe number to work with.")
    else:
        note("The two estimates differ by " + str(gap) + " tokens. Long or unusual words "
             "push the character estimate up. When in doubt, use the bigger number.")
    print("")
    print("Why it matters: you are billed per token, in both directions, on every")
    print("message - and the window is measured in tokens too.")

---
## Part 2: Will it actually fit?

Every model has a window size. Your bundle must fit inside it — **and leave room for the
reply**, because the answer is counted against the same budget.

In [ ]:
#@title ▶ Step 2 — Check whether my document fits { display-mode: "form" }
#@markdown How big is the thing you want to send?
document_size = "a long article (4,000 words)" #@param ["a chat message (30 words)", "a page (500 words)", "a long article (4,000 words)", "a report (20,000 words)", "a short book (60,000 words)"]
#@markdown Which window are you working with?
window = "medium (8,000 tokens)" #@param ["small (4,000 tokens)", "medium (8,000 tokens)", "large (32,000 tokens)", "very large (128,000 tokens)"]
#@markdown How much room should I keep aside for the reply?
reserve_for_reply = 800 #@param {type:"slider", min:100, max:4000, step:100}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

SIZES = {"a chat message (30 words)": 30, "a page (500 words)": 500,
         "a long article (4,000 words)": 4000, "a report (20,000 words)": 20000,
         "a short book (60,000 words)": 60000}
words = SIZES[document_size]
limit = WINDOWS[window]
need = round(words * 1.3)
usable = limit - reserve_for_reply
print("Your document : " + format(words, ",") + " words  ->  about " + format(need, ",") + " tokens")
print("The window    : " + format(limit, ",") + " tokens")
print("Kept for reply: " + format(reserve_for_reply, ",") + " tokens")
print("Room for you  : " + format(usable, ",") + " tokens")
print("")
print("   " + bar(need, limit) + "  " + pct(need, limit) + " of the window")
print("")
if usable <= 0:
    problem("You reserved " + format(reserve_for_reply, ",") + " tokens for the reply, which is the whole window.")
    note("Drag the reserve slider down, then press play again.")
elif need <= usable:
    success("It fits, with " + format(usable - need, ",") + " tokens to spare.")
    print("")
    print("You could send this in one go.")
else:
    over = need - usable
    problem("It does not fit. You are " + format(over, ",") + " tokens over - about "
            + format(round(over / 1.3), ",") + " words too many.")
    print("")
    print("What you would have to do instead:")
    print("   - chop it into chunks and send only the relevant ones  (Week 2.1 + 2.4)")
    print("   - or summarise it first, and send the summary")
    print("   - or move to a bigger window, and pay more per message")
    fits = round(usable / 1.3)
    note("At this window size you can send about " + format(fits, ",") + " words at once.")

---
## Part 3: The conversation that grows

Here is the trap. One message is small. But you re-send the **whole history** every time,
so each new turn makes the next one bigger. A chat that started tiny hits the wall by itself.

**Our sample conversation** (15 turns, you and your twin):

```
you : Hi! I am Sarah and I live in Accra.
twin: Hello Sarah! Nice to meet you. What do you do in Accra?
you : I run every morning before work. My best 5km time is 25 minutes.
...
you : Exactly. Now, what was my best 5km time again?
```

In [ ]:
#@title ▶ Step 3 — Watch the bundle grow with every turn { display-mode: "form" }
#@markdown How many turns of the conversation have happened so far?
turns_so_far = 15 #@param {type:"slider", min:1, max:15, step:1}
#@markdown Pretend the window is this small, so we can see the wall:
pretend_window = 220 #@param {type:"slider", min:80, max:600, step:20}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

SYSTEM = "You are Sarah's digital twin. Answer as she would: direct, warm, a little dry."
sys_cost = count_tokens(SYSTEM)
print("The standing instruction costs " + str(sys_cost) + " tokens and is re-sent EVERY time:")
show_text("", SYSTEM)
print("")
print("Turn | this message | whole bundle re-sent | window used")
print("-" * 64)
running = sys_cost
hit = None
for i, (who, msg) in enumerate(TURNS[:turns_so_far], 1):
    cost = count_tokens(msg)
    running += cost
    flag = ""
    if running > pretend_window and hit is None:
        hit = i; flag = "  <-- does not fit any more"
    print(str(i).rjust(4) + " | " + str(cost).rjust(12) + " | " + str(running).rjust(20)
          + " | " + pct(running, pretend_window).rjust(7) + flag)
print("")
print("   " + bar(running, pretend_window) + "  " + str(running) + " / " + str(pretend_window) + " tokens")
print("")
if hit:
    problem("The bundle outgrew the window at turn " + str(hit) + ".")
    print("")
    print("From that point on, every new message makes it worse. Something must go -")
    print("and if you do not choose what, the system chooses for you.")
    note("Step 4 shows the three usual ways to choose.")
else:
    success("Still fits: " + str(running) + " of " + str(pretend_window) + " tokens used.")
    print("")
    print("Drag 'turns so far' up, or the pretend window down, and press play again")
    print("to find the exact turn where it breaks.")

---
## Part 4: Three ways to make it fit

When the bundle is too big you have three real options. Each one keeps something and
loses something — there is no free choice.

| Strategy | Keeps | Loses |
|---|---|---|
| **Keep the last few turns** | what was just said | everything from early on |
| **Summarise the old turns** | the gist of everything | exact wording and detail |
| **Keep the first and last** | how the chat started, and where it is now | the middle |

In [ ]:
#@title ▶ Step 4 — Try a strategy and see what survives { display-mode: "form" }
#@markdown How should the history be trimmed?
strategy = "keep the last few turns" #@param ["keep the last few turns", "summarise the old turns", "keep the first and the last", "send everything (no trimming)"]
#@markdown How many recent turns to keep in full:
keep_recent = 4 #@param {type:"slider", min:1, max:12, step:1}
#@markdown Pretend the window is this small:
pretend_window = 220 #@param {type:"slider", min:80, max:600, step:20}
#@markdown The question being asked right now:
the_question = "What was my best 5km time again?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

SYSTEM = "You are Sarah's digital twin. Answer as she would: direct, warm, a little dry."
history = TURNS[:-1]
kept, dropped, summary = [], [], None
if strategy == "send everything (no trimming)":
    kept = list(history)
elif strategy == "keep the last few turns":
    kept = history[-keep_recent:]; dropped = history[:-keep_recent]
elif strategy == "keep the first and the last":
    kept = history[:2] + history[-keep_recent:]; dropped = history[2:-keep_recent]
else:
    kept = history[-keep_recent:]; dropped = history[:-keep_recent]
    facts = [msg.rstrip(".") for who, msg in dropped if who == "you"]
    summary = ("Earlier, Sarah said: " + "; ".join(facts[:6]) + ".") if facts else None

bundle = count_tokens(SYSTEM) + count_tokens(the_question)
if summary: bundle += count_tokens(summary)
for who, msg in kept: bundle += count_tokens(msg)

print("Strategy: " + strategy)
print("")
if summary:
    print("The dropped turns, squashed into one line (" + str(count_tokens(summary)) + " tokens):")
    show_text("", summary)
    print("")
if dropped and not summary:
    print("Thrown away completely (" + str(len(dropped)) + " turns):")
    for who, msg in dropped:
        print("   " + who.ljust(4) + " : " + msg[:62] + ("..." if len(msg) > 62 else ""))
    print("")
print("Kept in full (" + str(len(kept)) + " turns):")
for who, msg in kept:
    print("   " + who.ljust(4) + " : " + msg[:62] + ("..." if len(msg) > 62 else ""))
print("")
print("Plus the question: " + the_question)
print("")
print("   " + bar(bundle, pretend_window) + "  " + str(bundle) + " / " + str(pretend_window) + " tokens")
print("")
if bundle > pretend_window:
    problem("Still too big by " + str(bundle - pretend_window) + " tokens.")
    note("Keep fewer recent turns, or try summarising, then press play again.")
else:
    success("Fits, with " + str(pretend_window - bundle) + " tokens to spare.")

needle = "25 minutes"
visible = " ".join(m for _, m in kept) + (" " + summary if summary else "")
print("")
print("Now the real test - can it still answer \"" + the_question + "\"?")
if needle in visible:
    success("Yes. The answer (\"" + needle + "\") is still in what you sent.")
else:
    problem("No. \"" + needle + "\" was in a turn you threw away.")
    print("")
    print("Your twin will either guess, or admit it does not know - and you")
    print("will not be able to tell which from the reply.")
    note("Try 'summarise the old turns', or keep more recent turns, then press play again.")

---
## Part 5: The sliding window

The most common setup in a real product: a **pinned** instruction that never moves, plus a
window of recent turns that slides forward as the chat goes on. Old turns fall off the back.

In [ ]:
#@title ▶ Step 5 — Slide the window along the conversation { display-mode: "form" }
#@markdown Where in the conversation are we right now?
at_turn = 12 #@param {type:"slider", min:1, max:15, step:1}
#@markdown How many turns does the window hold?
window_holds = 5 #@param {type:"slider", min:1, max:10, step:1}
#@markdown Pin the standing instruction so it can never fall off?
pin_the_instruction = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

SYSTEM = "You are Sarah's digital twin. Answer as she would: direct, warm, a little dry."
upto = TURNS[:at_turn]
inside = upto[-window_holds:]
fallen = upto[:-window_holds]
print("Conversation so far: " + str(len(upto)) + " turns. The window holds " + str(window_holds) + ".")
print("")
for i, (who, msg) in enumerate(upto, 1):
    mark = "  in window  " if (who, msg) in inside and i > len(fallen) else "  fallen off "
    print(str(i).rjust(3) + mark + "| " + who.ljust(4) + " : " + msg[:52] + ("..." if len(msg) > 52 else ""))
print("")
cost = sum(count_tokens(m) for _, m in inside)
if pin_the_instruction:
    cost += count_tokens(SYSTEM)
    success("Pinned instruction: \"" + SYSTEM[:52] + "...\" - always sent, never falls off.")
else:
    problem("The instruction is NOT pinned, so it fell off with the old turns.")
    print("")
    print("Your twin has forgotten who it is supposed to be. It will answer like")
    print("a generic assistant from here on.")
    note("Tick 'pin the instruction' and press play again.")
print("")
print("What gets sent now: " + str(len(inside)) + " turns"
      + (" + the pinned instruction" if pin_the_instruction else "")
      + "  =  " + str(cost) + " tokens")
print("")
print("Notice the cost stops growing. However long the chat gets, the bundle")
print("stays the same size - that is the whole point of a sliding window.")
if fallen:
    print("")
    note(str(len(fallen)) + " turn(s) have fallen off the back and are gone unless you saved them elsewhere.")

---
## Part 6: Write your twin's memory policy

Every real assistant has a policy, whether it was written down or not. Write yours
deliberately: what is pinned forever, what slides, and what gets squashed into a summary.

In [ ]:
#@title ▶ Step 6 — Set my twin's memory policy { display-mode: "form" }
#@markdown The one thing your twin must never forget:
never_forget = "Sarah runs every morning; best 5km is 25 minutes; training for a 10km in March." #@param {type:"string"}
#@markdown How many recent turns stay in full?
keep_recent = 6 #@param {type:"slider", min:1, max:20, step:1}
#@markdown Squash older turns into a running summary?
summarise_the_rest = True #@param {type:"boolean"}
#@markdown Which window are you buying?
window = "medium (8,000 tokens)" #@param ["small (4,000 tokens)", "medium (8,000 tokens)", "large (32,000 tokens)", "very large (128,000 tokens)"]
#@markdown Room kept for the reply:
reserve_for_reply = 800 #@param {type:"slider", min:100, max:4000, step:100}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
WINDOWS = {
 "small (4,000 tokens)": 4000,
 "medium (8,000 tokens)": 8000,
 "large (32,000 tokens)": 32000,
 "very large (128,000 tokens)": 128000,
}
def count_tokens(text):
    """The rule of thumb everyone uses: about 4 characters per token."""
    return max(1, round(len(text) / 4))
def bar(part, whole, width=46):
    if whole <= 0: return "-" * width
    filled = min(width, int(round(width * part / whole)))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def pct(part, whole):
    return str(round(100.0 * part / whole, 1)) + "%" if whole else "?"
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
TURNS = [
 ("you",  "Hi! I am Sarah and I live in Accra."),
 ("twin", "Hello Sarah! Nice to meet you. What do you do in Accra?"),
 ("you",  "I run every morning before work. My best 5km time is 25 minutes."),
 ("twin", "That is a strong time. Are you training for anything in particular?"),
 ("you",  "Yes, a 10km race in March. I also code in the evenings."),
 ("twin", "A 10km in March, and evening coding. What language do you use?"),
 ("you",  "Mostly Python. I am learning to build a digital twin of myself."),
 ("twin", "That is a great project. What should your twin know about you first?"),
 ("you",  "That I am disciplined about mornings and I hate being late."),
 ("twin", "Noted - disciplined in the mornings, and punctual. Anything else?"),
 ("you",  "I drink too much coffee. Three cups before noon, usually."),
 ("twin", "Three cups before noon, noted. Does the coffee help the running?"),
 ("you",  "It helps the coding more than the running, honestly."),
 ("twin", "Understood. So: coffee for code, discipline for the morning runs."),
 ("you",  "Exactly. Now, what was my best 5km time again?"),
]

limit = WINDOWS[window]
usable = limit - reserve_for_reply
pinned = count_tokens(never_forget)
avg = round(sum(count_tokens(m) for _, m in TURNS) / len(TURNS))
recent = keep_recent * avg
summary = 120 if summarise_the_rest else 0
total = pinned + recent + summary
if not never_forget.strip():
    problem("Write down the one thing your twin must never forget, then press play again.")
elif usable <= 0:
    problem("You reserved the entire window for the reply. Drag that slider down and press play again.")
else:
    print("YOUR TWIN'S MEMORY POLICY")
    print("=" * 56)
    print("")
    print("1. PINNED - sent every single time, never dropped:")
    show_text("", never_forget)
    print("   cost: " + str(pinned) + " tokens per message")
    print("")
    print("2. RECENT - the last " + str(keep_recent) + " turns, word for word")
    print("   cost: about " + str(recent) + " tokens (at " + str(avg) + " tokens a turn)")
    print("")
    if summarise_the_rest:
        print("3. SUMMARY - everything older, squashed into a running paragraph")
        print("   cost: about " + str(summary) + " tokens, and it stops growing")
    else:
        print("3. OLDER TURNS - discarded, nothing kept")
        print("   cost: 0 tokens, but anything said early on is gone for good")
    print("")
    print("=" * 56)
    print("Steady cost per message: about " + str(total) + " tokens, out of "
          + str(usable) + " usable.")
    print("")
    print("   " + bar(total, usable) + "  " + pct(total, usable) + " of your usable window")
    print("")
    if total > usable:
        problem("This policy does not fit - it is " + str(total - usable) + " tokens over.")
        note("Keep fewer recent turns, or choose a bigger window, then press play again.")
    else:
        success("This policy fits, and it stays the same size however long the chat runs.")
        print("")
        print("Room left over for retrieved notes and documents: "
              + str(usable - total) + " tokens.")
        if not summarise_the_rest:
            note("Without a summary, anything from early in the conversation is lost "
                 "the moment it falls out of the recent turns.")

---
## What you just did

| Step | In plain words | Why it matters |
|---|---|---|
| 1 | Measured text in tokens | Tokens are what you are billed and limited by |
| 2 | Checked whether a document fits | Most documents do not — that is why chunking exists |
| 3 | Watched a conversation grow | Long chats break on their own, with no warning |
| 4 | Compared three trimming strategies | Every strategy loses something; you choose what |
| 5 | Slid the window along | Keeps cost flat however long the chat runs |
| 6 | Wrote a memory policy | Pinned / recent / summarised — the real design decision |

### The uncomfortable bit
When a fact falls out of the window, your twin does not say *"I used to know that."*
It just answers anyway. **A confident answer is not evidence that the fact was still there.**

### Where this is going
If you cannot send everything, you must send *the right part* — which means being able to
find it. That needs text turned into numbers (**Week 2.3: Embeddings**), and a way to search
with them (**Week 2.4: RAG**).

### Want to see the code?
Click **Show code** on any step.